# 🚀 Workshop Day 1: Data Engineering — Turning Junk into Fuel
## Agentic RAG: From Zero to Hero

**เวลา:** 3 ชั่วโมง | **ระดับ:** ปริญญาตรี

---

### 🎯 เป้าหมายของวันนี้

เราจะเรียนรู้ **Data Engineering Pipeline** สำหรับ RAG ตั้งแต่ต้นจนจบ:

```
📄 เอกสารดิบ → 🔍 ตรวจ Duplicate → ✂️ Chunking → 📝 Markdown → 🔢 Embedding → 🗄️ VectorDB → 🔎 Retrieval
```

> **RAG (Retrieval-Augmented Generation)** คือเทคนิคที่ช่วยให้ LLM ตอบคำถามได้แม่นยำขึ้น
> โดยการ "ค้นหา" ข้อมูลที่เกี่ยวข้องมาให้ก่อนที่จะ "สร้าง" คำตอบ


### 🗺️ ภาพรวม Pipeline วันนี้

```
📁 Raw Documents (PDF, TXT, DOCX)
       │
       ▼
┌─────────────────┐
│ 1.1 Deduplication│  ← ตรวจไฟล์ซ้ำด้วย MD5 Hash
└────────┬────────┘
         ▼
┌─────────────────┐
│ 1.2 Chunking     │  ← ตัดข้อความเป็นส่วนย่อย
└────────┬────────┘
         ▼
┌─────────────────┐
│ 1.3-1.4 Markdown │  ← แปลง PDF → Markdown (Gemini / Docling)
└────────┬────────┘
         ▼
┌─────────────────┐
│ 1.5 Embedding    │  ← แปลงข้อความ → Vector (multilingual-e5-large)
└────────┬────────┘
         ▼
┌─────────────────┐
│ 1.6 Hybrid Search│  ← ผสม Dense + Sparse (BM25)
└────────┬────────┘
         ▼
┌─────────────────┐
│ 1.7 VectorDB     │  ← ติดตั้ง Qdrant
└────────┬────────┘
         ▼
┌─────────────────┐
│ 1.8 Indexing     │  ← Upsert vectors เข้า Qdrant
└────────┬────────┘
         ▼
┌─────────────────┐
│ 1.9 Retrieval    │  ← ค้นหา vectors ที่คล้ายกับ query
└─────────────────┘
```

> 💡 วันนี้เราจะทำ **ทุกขั้นตอน** ตั้งแต่เตรียมข้อมูลจนถึงค้นหา — เป็นพื้นฐานของ RAG ทั้งหมด!

---
## 📦 Section 0: ติดตั้ง Dependencies

ติดตั้ง library ที่จำเป็นทั้งหมดสำหรับ workshop วันนี้

In [ ]:
%%time
# ติดตั้ง libraries ทั้งหมด
!pip install -q google-genai \
    docling \
    'typer>=0.24.0' \
    sentence-transformers \
    qdrant-client \
    langchain-text-splitters \
    rank_bm25 \
    pymupdf \
    pythainlp \
    scikit-learn \
    rich

print('✅ ติดตั้งเรียบร้อยแล้ว!')

In [ ]:
%%time
# Import libraries หลัก
import hashlib
import os
import json
import numpy as np
from pathlib import Path
from IPython.display import display, Markdown

print('✅ Import สำเร็จ!')

### 📂 เตรียมข้อมูลตัวอย่าง

เราจะสร้างข้อมูลตัวอย่างภาษาไทยสำหรับใช้ตลอด workshop

In [ ]:
%%time
# สร้างโฟลเดอร์สำหรับเก็บข้อมูล
os.makedirs('data', exist_ok=True)
os.makedirs('output', exist_ok=True)

# ข้อมูลตัวอย่าง: สรุป Case Study ด้าน AI ในประเทศไทย
sample_texts = {

    'case_healthcare.txt': '''กรณีศึกษา: AI ในการแพทย์ไทย

โรงพยาบาลศิริราชได้นำ AI มาใช้ในการวิเคราะห์ภาพถ่ายทางการแพทย์ (Medical Imaging)
เช่น การตรวจจับมะเร็งปอดจากภาพ X-ray ด้วย Deep Learning
ผลการทดสอบพบว่า AI มีความแม่นยำ 95% เทียบกับรังสีแพทย์ที่ 92%

นอกจากนี้ยังมีการใช้ NLP วิเคราะห์เวชระเบียนอิเล็กทรอนิกส์ (EMR)
เพื่อช่วยแพทย์สรุปประวัติผู้ป่วยและแนะนำการรักษาที่เหมาะสม
ลดเวลาการอ่านเวชระเบียนจาก 15 นาทีเหลือ 2 นาทีต่อเคส

ความท้าทาย: ข้อมูลทางการแพทย์ภาษาไทยมีจำนวนจำกัด
ต้องใช้เทคนิค Transfer Learning จากโมเดลภาษาอังกฤษ
และ Fine-tune ด้วยข้อมูลภาษาไทยเพิ่มเติม''',

    'case_banking.txt': '''กรณีศึกษา: AI ในธนาคารและการเงิน

ธนาคารกสิกรไทยได้พัฒนาระบบ Chatbot ชื่อ KBTG
ที่ใช้ Large Language Model (LLM) ร่วมกับ RAG
ในการตอบคำถามลูกค้าเกี่ยวกับผลิตภัณฑ์ทางการเงิน

ระบบ RAG ทำงานโดยการค้นหาข้อมูลจากฐานความรู้ภายใน
ซึ่งประกอบด้วยเอกสารผลิตภัณฑ์ เงื่อนไขบริการ และ FAQ
จากนั้น LLM จะสร้างคำตอบที่เป็นธรรมชาติจากข้อมูลที่ค้นพบ

ผลลัพธ์: ลดภาระ Call Center ได้ 40%
ความพึงพอใจลูกค้าเพิ่มขึ้น 25%
สามารถให้บริการ 24 ชั่วโมง โดยไม่ต้องรอพนักงาน

เทคโนโลยีที่ใช้: Vector Database สำหรับเก็บ Embedding
Hybrid Search ผสม Dense + Sparse เพื่อค้นหาข้อมูลที่ตรงกับคำถาม''',

    'case_banking_duplicate.txt': '''กรณีศึกษา: AI ในธนาคารและการเงิน

ธนาคารกสิกรไทยได้พัฒนาระบบ Chatbot ชื่อ KBTG
ที่ใช้ Large Language Model (LLM) ร่วมกับ RAG
ในการตอบคำถามลูกค้าเกี่ยวกับผลิตภัณฑ์ทางการเงิน

ระบบ RAG ทำงานโดยการค้นหาข้อมูลจากฐานความรู้ภายใน
ซึ่งประกอบด้วยเอกสารผลิตภัณฑ์ เงื่อนไขบริการ และ FAQ
จากนั้น LLM จะสร้างคำตอบที่เป็นธรรมชาติจากข้อมูลที่ค้นพบ

ผลลัพธ์: ลดภาระ Call Center ได้ 40%
ความพึงพอใจลูกค้าเพิ่มขึ้น 25%
สามารถให้บริการ 24 ชั่วโมง โดยไม่ต้องรอพนักงาน

เทคโนโลยีที่ใช้: Vector Database สำหรับเก็บ Embedding
Hybrid Search ผสม Dense + Sparse เพื่อค้นหาข้อมูลที่ตรงกับคำถาม''',

    'case_education.txt': '''กรณีศึกษา: AI ในการศึกษาไทย

มหาวิทยาลัยหลายแห่งในไทยเริ่มนำ AI มาช่วยในการเรียนการสอน
เช่น ระบบ Intelligent Tutoring System ที่ปรับเนื้อหาตามระดับของผู้เรียน

ตัวอย่างที่น่าสนใจคือการใช้ RAG สร้างระบบถาม-ตอบอัตโนมัติ
สำหรับวิชาเรียน โดยนำเอกสารประกอบการสอน Slides และหนังสือ
มา Embed เป็น Vector แล้วเก็บใน Vector Database

เมื่อนักศึกษาถามคำถาม ระบบจะค้นหาเนื้อหาที่เกี่ยวข้อง
แล้วใช้ LLM สร้างคำตอบพร้อมอ้างอิงแหล่งที่มา

ขั้นตอนสำคัญในการสร้างระบบนี้:
1. เตรียมข้อมูล: แปลง PDF เป็น Markdown
2. ตัดข้อความ: ใช้ Chunking แบ่งเนื้อหาเป็นส่วนย่อย
3. สร้าง Embedding: แปลง Chunk เป็น Vector ด้วยโมเดลที่รองรับภาษาไทย
4. จัดเก็บ: Upsert Vector เข้า Qdrant หรือ Vector DB อื่น
5. ค้นหา: ใช้ Hybrid Search ค้นหา Chunk ที่เกี่ยวข้อง
6. สร้างคำตอบ: ส่ง Context ให้ LLM สร้างคำตอบ

ผลลัพธ์: นักศึกษาสามารถเรียนรู้ด้วยตนเองได้ตลอด 24 ชั่วโมง
ลดภาระอาจารย์ในการตอบคำถามซ้ำๆ ได้กว่า 60%''',

    'case_agriculture.txt': '''กรณีศึกษา: AI ในการเกษตรอัจฉริยะ

Smart Farming ในประเทศไทยใช้ AI วิเคราะห์ภาพถ่ายดาวเทียม
และข้อมูลจาก IoT Sensor เพื่อพยากรณ์ผลผลิตและเฝ้าระวังโรคพืช

ระบบ Computer Vision สามารถตรวจจับโรคข้าวได้จากภาพถ่ายใบข้าว
โดยใช้ Convolutional Neural Network (CNN) ที่ Train ด้วยภาพถ่าย
โรคข้าวกว่า 50,000 ภาพ ครอบคลุม 8 โรคหลัก

นอกจากนี้ยังมีการใช้ NLP วิเคราะห์ข้อมูลราคาสินค้าเกษตร
จากข่าวสาร รายงานภาครัฐ และ Social Media
เพื่อช่วยเกษตรกรตัดสินใจเรื่องการปลูกและการขาย

ความท้าทายของ AI ในการเกษตรไทย:
- ข้อมูลกระจัดกระจายอยู่ในหลายหน่วยงาน
- ภาษาและศัพท์เฉพาะทางการเกษตรไทย
- ต้องทำ Data Engineering เพื่อรวมและทำความสะอาดข้อมูลก่อน'''
}

for fname, content in sample_texts.items():
    with open(f'data/{fname}', 'w', encoding='utf-8') as f:
        f.write(content)

print(f'✅ สร้างไฟล์ตัวอย่าง {len(sample_texts)} ไฟล์ ในโฟลเดอร์ data/')
print()
for fname in sorted(sample_texts.keys()):
    print(f'  📄 {fname}')

---
## 🔍 Section 1.1: ตรวจสอบ Duplicate ด้วย MD5

### MD5 Hash คืออะไร?

**MD5 (Message Digest Algorithm 5)** คืออัลกอริทึมที่แปลงข้อมูลใดๆ ให้เป็นค่า hash ขนาด 128-bit (32 ตัวอักษร hex)

- ข้อมูลเดียวกัน → ได้ hash เดียวกัน **เสมอ**
- ข้อมูลต่างกันแม้เพียงนิดเดียว → ได้ hash ต่างกัน **โดยสิ้นเชิง**

### ทำไมต้องเช็ค Duplicate?

ในงาน RAG ถ้ามีเอกสารซ้ำกัน:
- ❌ เปลือง storage และเวลา embedding
- ❌ ผลการค้นหาซ้ำซ้อน
- ❌ LLM อาจสับสนจากบริบทที่ซ้ำกัน

In [ ]:
%%time
def compute_md5(filepath):
    """คำนวณ MD5 hash ของไฟล์"""
    md5_hash = hashlib.md5()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(4096), b''):
            md5_hash.update(chunk)
    return md5_hash.hexdigest()

# ทดสอบ: คำนวณ MD5 ของทุกไฟล์
print('📊 MD5 Hash ของแต่ละไฟล์:')
print('=' * 70)
file_hashes = {}
for fname in sorted(os.listdir('data')):
    fpath = f'data/{fname}'
    if os.path.isfile(fpath):
        h = compute_md5(fpath)
        file_hashes[fname] = h
        print(f'  {fname:<25} → {h}')

print('\n🔍 สังเกต: case_banking.txt และ case_banking_duplicate.txt มี hash เดียวกัน!')

In [ ]:
%%time
def find_duplicates(directory):
    """ค้นหาไฟล์ที่ซ้ำกันในโฟลเดอร์โดยใช้ MD5"""
    hash_map = {}  # md5 → [list of filepaths]
    
    for fname in os.listdir(directory):
        fpath = os.path.join(directory, fname)
        if os.path.isfile(fpath):
            h = compute_md5(fpath)
            hash_map.setdefault(h, []).append(fname)
    
    # กรองเฉพาะกลุ่มที่มีมากกว่า 1 ไฟล์
    duplicates = {h: files for h, files in hash_map.items() if len(files) > 1}
    return duplicates, hash_map

duplicates, hash_map = find_duplicates('data')

if duplicates:
    print('⚠️ พบไฟล์ซ้ำกัน!')
    for h, files in duplicates.items():
        print(f'\n  Hash: {h}')
        print(f'  ไฟล์ที่ซ้ำกัน: {", ".join(files)}')
        print(f'  → เก็บไว้: {files[0]}, ลบออก: {", ".join(files[1:])}')
else:
    print('✅ ไม่พบไฟล์ซ้ำกัน')

In [ ]:
%%time
# ลบไฟล์ที่ซ้ำ (เก็บเฉพาะตัวแรก)
removed = []
for h, files in duplicates.items():
    for f in files[1:]:  # ลบทุกตัวยกเว้นตัวแรก
        os.remove(f'data/{f}')
        removed.append(f)

print(f'🗑️ ลบไฟล์ซ้ำแล้ว {len(removed)} ไฟล์: {", ".join(removed)}')
print(f'📁 ไฟล์ที่เหลือ: {sorted(os.listdir("data"))}')

### 🧪 แบบฝึกหัด 1.1

ลองสร้างไฟล์เพิ่มอีก 2-3 ไฟล์ (มีบางไฟล์ซ้ำ) แล้วใช้ฟังก์ชัน `find_duplicates()` ตรวจสอบ

In [ ]:
%%time
# 🧪 แบบฝึกหัด: สร้างไฟล์เพิ่มและทดสอบ duplicate detection
# TODO: เขียนโค้ดของคุณที่นี่



---
## ✂️ Section 1.2: Chunking แบบต่างๆ

### ทำไมต้อง Chunk?

- LLM มี **context window จำกัด** — ส่งเอกสาร 100 หน้าทั้งหมดไม่ได้
- Embedding model ทำงานได้ดีกับข้อความ **สั้นถึงปานกลาง** (ไม่เกิน 512 tokens)
- Chunk ที่เหมาะสมช่วยให้ **retrieval แม่นยำขึ้น**

### วิธีการ Chunk ที่เราจะเรียนรู้

| วิธี | หลักการ | ข้อดี | ข้อเสีย |
|------|---------|-------|--------|
| Fixed-size | ตัดตามจำนวนตัวอักษร | ง่าย, เร็ว | อาจตัดกลางประโยค |
| Recursive | แบ่งตาม separator ซ้อนกัน | ผลดีกว่า fixed | ต้องกำหนด separators |
| Sentence-based | ตัดตามประโยค | รักษาความหมาย | ขนาด chunk ไม่สม่ำเสมอ |
| Semantic | ใช้ embedding จับกลุ่ม | แม่นยำที่สุด | ช้า, ต้นทุนสูง |

In [ ]:
%%time
# โหลดข้อความตัวอย่าง
all_docs = {}  # เก็บแต่ละเอกสารแยกกัน
all_text = ''

for fname in sorted(os.listdir('data')):
    fpath = f'data/{fname}'
    if os.path.isfile(fpath) and fname.endswith('.txt'):
        with open(fpath, 'r', encoding='utf-8') as f:
            content = f.read()
            all_docs[fname] = content
            all_text += '\n\n' + content

print(f'📄 จำนวนเอกสาร: {len(all_docs)} ไฟล์')
print(f'📄 ข้อความรวม: {len(all_text)} ตัวอักษร')
for fname, content in all_docs.items():
    print(f'  📄 {fname}: {len(content)} ตัวอักษร')

### วิธีที่ 1: Fixed-size Chunking

In [ ]:
%%time
def fixed_size_chunk(text, chunk_size=200, overlap=50):
    """ตัดข้อความตามจำนวนตัวอักษรที่กำหนด"""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap  # เลื่อนกลับมาซ้อนเพื่อไม่ให้ขาดบริบท
    return chunks

fixed_chunks = fixed_size_chunk(all_text, chunk_size=200, overlap=50)
print(f'📊 Fixed-size: ได้ {len(fixed_chunks)} chunks')
print(f'\n--- Chunk ที่ 1 ---')
print(fixed_chunks[0])
print(f'\n--- Chunk ที่ 2 ---')
print(fixed_chunks[1])

### วิธีที่ 2: Recursive Character Chunking (LangChain)

วิธีนี้จะพยายาม **แบ่งด้วย separator ตัวแรกก่อน** ถ้า chunk ยังใหญ่เกินไป → ลองตัวถัดไป

```python
separators = ['\n\n', '\n', '。', r'\. ', ' ', '']
#              ①      ②    ③     ④    ⑤   ⑥
```

| ลำดับ | Separator | ความหมาย | ตัวอย่าง |
|:-----:|-----------|----------|----------|
| ① | `\n\n` | **บรรทัดว่าง** (ย่อหน้าใหม่) | แบ่งระหว่างย่อหน้า |
| ② | `\n` | **ขึ้นบรรทัดใหม่** | แบ่งทีละบรรทัด |
| ③ | `。` | **มหัพภาค (จุดจีน/ญี่ปุ่น)** | จบประโยคภาษา CJK |
| ④ | `\. ` | **จุดภาษาอังกฤษ + เว้นวรรค** | "Hello. World" → ตัดตรงจุด |
| ⑤ | ` ` | **เว้นวรรค** | แบ่งทีละคำ |
| ⑥ | `''` | **ทุกตัวอักษร** (ทางเลือกสุดท้าย) | ตัดตรงกลางคำ |

**หลักการทำงาน:**
```
ข้อความยาวมาก → ลองตัดด้วย \n\n ก่อน
                  ↓ ถ้า chunk ยังยาว
                  ลองตัดด้วย \n
                  ↓ ถ้ายังยาว
                  ลองตัดด้วย จุด
                  ↓ ...
                  ตัดทีละตัวอักษร (สุดท้าย)
```

> 💡 **สำหรับภาษาไทย**: ไม่มีเว้นวรรคระหว่างคำ ดังนั้น separator ` ` จะไม่ค่อยมีผล
> การแบ่งด้วย `\n\n` และ `\n` จะเป็นตัวหลัก

In [ ]:
%%time
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
    separators=['\n\n', '\n', '。', r'\. ', ' ', ''],  # ลำดับการแบ่ง
)

recursive_chunks = recursive_splitter.split_text(all_text)
print(f'📊 Recursive: ได้ {len(recursive_chunks)} chunks')
print(f'\n--- Chunk ที่ 1 ---')
print(recursive_chunks[0])
print(f'\n--- Chunk ที่ 2 ---')
print(recursive_chunks[1])

### วิธีที่ 3: Sentence-based Chunking

In [ ]:
%%time
import re

def sentence_chunk(text, max_sentences=3):
    """ตัดข้อความตามประโยค/บรรทัด (เหมาะกับภาษาไทย)"""
    # ภาษาไทยใช้ newline เป็นตัวแบ่งประโยคหลัก
    sentences = re.split(r'\n+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    
    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunk = '\n'.join(sentences[i:i+max_sentences])
        if chunk:
            chunks.append(chunk)
    return chunks

sentence_chunks = sentence_chunk(all_text, max_sentences=3)
print(f'📊 Sentence-based: ได้ {len(sentence_chunks)} chunks')

# แสดง 3 chunks แรก
for idx in range(min(3, len(sentence_chunks))):
    print(f'\n--- Chunk ที่ {idx+1} ({len(sentence_chunks[idx])} ตัวอักษร) ---')
    print(sentence_chunks[idx])

### 📊 เปรียบเทียบผลลัพธ์

In [ ]:
%%time
print('📊 เปรียบเทียบจำนวน Chunks:')
print(f'  Fixed-size (200 chars):     {len(fixed_chunks)} chunks')
print(f'  Recursive (200 chars):      {len(recursive_chunks)} chunks')
print(f'  Sentence-based (3 sent):    {len(sentence_chunks)} chunks')

print('\n📏 ขนาดเฉลี่ยของ Chunk:')
for name, chunks in [('Fixed', fixed_chunks), ('Recursive', recursive_chunks), ('Sentence', sentence_chunks)]:
    sizes = [len(c) for c in chunks]
    print(f'  {name:<12}: avg={np.mean(sizes):.0f}, min={min(sizes)}, max={max(sizes)} ตัวอักษร')

### 🧪 แบบฝึกหัด 1.2

ลองปรับค่า `chunk_size` และ `overlap` แล้วเปรียบเทียบผลลัพธ์ ค่าไหนเหมาะกับข้อความภาษาไทยมากที่สุด?

In [ ]:
%%time
# 🧪 แบบฝึกหัด: ทดลองปรับค่า chunk_size และ overlap
# TODO: เขียนโค้ดของคุณที่นี่



---
## 🤖 Section 1.3: แปลงเอกสารเป็น Markdown ด้วย Gemini

### ทำไมต้องแปลงเป็น Markdown?

- PDF, Word, PowerPoint มีรูปแบบที่ซับซ้อน
- Markdown เป็น **plain text** ที่ยังรักษาโครงสร้าง (หัวข้อ, ตาราง, รายการ)
- ง่ายต่อการ chunk และ embed

### ใช้ Gemini API

เราจะใช้ **Gemini** ซึ่งเป็น multimodal model ในการ "อ่าน" เอกสาร แล้วแปลงเป็น Markdown

In [ ]:
%%time
# ตั้งค่า Gemini API
# 🔑 ไปสร้าง API Key ได้ที่: https://aistudio.google.com/apikey
#
# วิธีตั้งค่า Colab Secrets:
#   1. คลิกไอคอน 🔑 ที่ sidebar ด้านซ้าย
#   2. กด 'Add new secret'
#   3. ตั้งชื่อ: GOOGLE_API_KEY
#   4. วาง API Key ที่ copy มา
#   5. เปิด toggle 'Notebook access'

try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    print('✅ โหลด API Key จาก Colab Secrets สำเร็จ!')
except Exception:
    GOOGLE_API_KEY = input('🔑 กรุณาวาง Gemini API Key ของคุณ: ')

from google import genai
client = genai.Client(api_key=GOOGLE_API_KEY)

# ทดสอบว่า API ใช้งานได้
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='ตอบว่า สวัสดี เท่านั้น'
)
print(f'✅ ทดสอบ Gemini API สำเร็จ: {response.text.strip()}')

### สร้าง Sample PDF สำหรับทดสอบ

เราจะสร้าง PDF ตัวอย่างจากข้อความภาษาไทย

In [ ]:
%%time
import fitz  # PyMuPDF
import subprocess

# ติดตั้ง Thai font บน Colab
subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-thai-tlwg'], 
               capture_output=True)
print('✅ ติดตั้ง Thai font สำเร็จ')

# หา path ของ font ภาษาไทย
import glob
thai_fonts = glob.glob('/usr/share/fonts/truetype/tlwg/*.ttf')
THAI_FONT = thai_fonts[0] if thai_fonts else None
print(f'📝 ใช้ font: {THAI_FONT}')

def create_sample_pdf(filepath, text, title='เอกสารตัวอย่าง'):
    """สร้าง PDF จากข้อความภาษาไทย"""
    doc = fitz.open()
    page = doc.new_page()
    
    # โหลด font ภาษาไทย
    thai_font = fitz.Font(fontfile=THAI_FONT)
    
    # เขียนหัวข้อ
    tw = fitz.TextWriter(page.rect)
    y = 50
    tw.append((50, y), title, font=thai_font, fontsize=18)
    y += 40
    
    # เขียนเนื้อหาทีละบรรทัด
    for line in text.split('\n'):
        if y > 750:  # ขึ้นหน้าใหม่
            tw.write_text(page)
            page = doc.new_page()
            tw = fitz.TextWriter(page.rect)
            y = 50
        tw.append((50, y), line, font=thai_font, fontsize=11)
        y += 18
    
    tw.write_text(page)
    doc.save(filepath)
    doc.close()

# อ่านข้อความมาสร้าง PDF
with open('data/case_healthcare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

create_sample_pdf('data/sample.pdf', text, 'กรณีศึกษา: AI ในการแพทย์ไทย')
print('✅ สร้าง sample.pdf สำเร็จ (รองรับภาษาไทย!)')

In [ ]:
%%time
# ใช้ Gemini แปลง PDF เป็น Markdown
import pathlib

def pdf_to_markdown_gemini(pdf_path, client):
    """ใช้ Gemini อ่าน PDF แล้วแปลงเป็น Markdown"""
    # อ่านไฟล์ PDF
    pdf_bytes = pathlib.Path(pdf_path).read_bytes()
    
    prompt = '''อ่านเอกสาร PDF นี้แล้วแปลงเนื้อหาทั้งหมดเป็น Markdown format
กฎ:
- รักษาโครงสร้างเดิม (หัวข้อ, ย่อหน้า, ตาราง)
- ใช้ heading (#, ##, ###) ตามลำดับความสำคัญ
- ถ้ามีตาราง ให้แปลงเป็น Markdown table
- ถ้ามีรายการ ให้ใช้ bullet points
- เก็บเนื้อหาครบถ้วน ห้ามสรุปหรือตัดทอน'''
    
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=[
            genai.types.Part.from_bytes(data=pdf_bytes, mime_type='application/pdf'),
            prompt
        ]
    )
    return response.text

# ทดสอบ
markdown_output = pdf_to_markdown_gemini('data/sample.pdf', client)
print('📝 ผลลัพธ์ Markdown จาก Gemini:')
print('=' * 60)
display(Markdown(markdown_output))

In [ ]:
%%time
# บันทึกผลลัพธ์
with open('output/gemini_output.md', 'w', encoding='utf-8') as f:
    f.write(markdown_output)
print('✅ บันทึกผลลัพธ์ไว้ที่ output/gemini_output.md')

### 🧪 แบบฝึกหัด 1.3

ลอง upload PDF ของตัวเอง (เช่น slides จากวิชาเรียน) แล้วใช้ Gemini แปลงเป็น Markdown

In [ ]:
%%time
# 🧪 แบบฝึกหัด: แปลง PDF ของตัวเอง
# TODO: Upload PDF ของคุณ แล้วใช้ pdf_to_markdown_gemini()



---
## 📄 Section 1.4: แปลงเอกสารเป็น Markdown ด้วย Docling

### Docling คืออะไร?

**[Docling](https://github.com/DS4SD/docling)** เป็น open-source library จาก IBM
สำหรับแปลงเอกสาร (PDF, DOCX, PPTX, HTML) เป็น Markdown หรือ JSON

| | Gemini | Docling |
|---|--------|--------|
| **ประเภท** | Cloud API (LLM) | Local library |
| **ค่าใช้จ่าย** | มี quota จำกัด | ฟรี ไม่จำกัด |
| **ความเร็ว** | ขึ้นกับ network | ขึ้นกับ CPU/GPU |
| **ข้อดี** | เข้าใจบริบทดี | ทำงาน offline ได้ |
| **ข้อเสีย** | ต้องมี internet | อาจไม่เข้าใจบริบทซ้อน |

In [ ]:
%%time
from docling.document_converter import DocumentConverter

# แปลง PDF ด้วย Docling
converter = DocumentConverter()
result = converter.convert('data/sample.pdf')

docling_markdown = result.document.export_to_markdown()

print('📝 ผลลัพธ์ Markdown จาก Docling:')
print('=' * 60)
display(Markdown(docling_markdown))

In [ ]:
%%time
# บันทึกเปรียบเทียบ
with open('output/docling_output.md', 'w', encoding='utf-8') as f:
    f.write(docling_markdown)

print('📊 เปรียบเทียบ Gemini vs Docling:')
print(f'  Gemini output:  {len(markdown_output)} ตัวอักษร')
print(f'  Docling output: {len(docling_markdown)} ตัวอักษร')
print('\n💡 Tip: ในโปรเจกต์จริง ควรเลือกใช้ตามสถานการณ์')
print('  - เอกสารซับซ้อน (กราฟ, ตาราง) → Gemini')
print('  - เอกสารจำนวนมาก, ต้อง offline → Docling')

### 📊 เปรียบเทียบ Gemini vs Docling

| เกณฑ์ | Gemini API | Docling (IBM) |
|-------|-----------|---------------|
| **ความเร็ว** | ⚡ เร็ว (ส่ง API) | 🐢 ช้ากว่า (run local) |
| **คุณภาพ Markdown** | ⭐⭐⭐ ดีมาก | ⭐⭐ ดี |
| **รองรับภาษาไทย** | ✅ ดีเยี่ยม | ⚠️ พอใช้ |
| **ราคา** | 💰 ฟรี (มี quota) | 🆓 ฟรีทั้งหมด |
| **ต้องการ Internet** | ✅ ต้อง | ❌ ไม่ต้อง |
| **รองรับตาราง/รูป** | ✅ ดี | ✅ ดีมาก |
| **ข้อมูลส่วนตัว** | ⚠️ ส่งขึ้น cloud | ✅ ประมวลผลในเครื่อง |
| **เหมาะกับ** | Prototype, เอกสารทั่วไป | Production, เอกสารลับ |

### 🧪 แบบฝึกหัด 1.4

ลองแปลง PDF เดียวกันด้วยทั้ง Gemini และ Docling แล้วเปรียบเทียบว่าวิธีไหนให้ผลลัพธ์ดีกว่า

In [ ]:
%%time
# 🧪 แบบฝึกหัด: เปรียบเทียบ Gemini vs Docling กับ PDF ของตัวเอง
# TODO: เขียนโค้ดของคุณที่นี่



---
## 🔢 Section 1.5: Embedding แบบปกติ (รองรับภาษาไทย)

### Text Embedding คืออะไร?

**Embedding** คือการแปลงข้อความเป็น **เวกเตอร์** (ชุดตัวเลข) ที่เก็บ "ความหมาย" ของข้อความ

```
"แมวนอนบนโซฟา" → [0.12, -0.34, 0.56, 0.78, ...] (1024 มิติ)
"น้องแมวนอนพักผ่อน" → [0.11, -0.33, 0.55, 0.77, ...] ← คล้ายกัน!
"วันนี้ฝนตก" → [0.89, 0.22, -0.67, 0.11, ...] ← ต่างกัน!
```

### ทำไมต้องใช้โมเดลที่รองรับภาษาไทย?

โมเดลที่ train เฉพาะภาษาอังกฤษจะไม่เข้าใจความหมายภาษาไทย
เราจะใช้ **`multilingual-e5-large`** ที่รองรับ 100+ ภาษา รวมถึงภาษาไทย

In [ ]:
%%time
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# โหลดโมเดล multilingual
print('⏳ กำลังโหลดโมเดล embedding...')
embed_model = SentenceTransformer('intfloat/multilingual-e5-large')
print(f'✅ โหลดโมเดลสำเร็จ! (ขนาด vector: {embed_model.get_sentence_embedding_dimension()} มิติ)')

In [ ]:
%%time
# ทดสอบ embedding ภาษาไทย
thai_sentences = [
    'query: ปัญญาประดิษฐ์คืออะไร',
    'passage: AI คือสาขาของวิทยาการคอมพิวเตอร์ที่สร้างระบบอัจฉริยะ',
    'passage: Machine Learning เป็นส่วนหนึ่งของ AI ที่เรียนรู้จากข้อมูล',
    'passage: วันนี้อากาศดีมาก ท้องฟ้าแจ่มใส',
    'passage: Vector Database ใช้เก็บข้อมูลแบบเวกเตอร์',
]

# สร้าง embeddings
embeddings = embed_model.encode(thai_sentences)

print(f'📊 สร้าง embedding สำเร็จ!')
print(f'  จำนวนข้อความ: {len(embeddings)}')
print(f'  ขนาด vector: {embeddings.shape[1]} มิติ')
print(f'  ตัวอย่าง vector (5 ค่าแรก): {embeddings[0][:5]}')

### 📐 วิธีวัดความคล้ายคลึงของ Vectors

เมื่อเราแปลงข้อความเป็น Vector แล้ว ขั้นตอนต่อไปคือ **เปรียบเทียบ** ว่า vector ไหนคล้ายกัน
มี 3 วิธีหลักที่นิยมใช้:

---

**1️⃣ Cosine Similarity** — วัด "มุม" ระหว่าง 2 vectors
```
                    A · B           ผลรวมของ (a₁×b₁ + a₂×b₂ + ...)
Cosine(A, B) = ─────────── = ─────────────────────────────────────────
                |A| × |B|       ความยาวของ A × ความยาวของ B

ค่า:  -1 (ตรงข้าม) ← 0 (ไม่เกี่ยว) → 1 (คล้ายมาก) ✅
```

**2️⃣ Dot Product** — คูณค่าแต่ละมิติแล้วรวมกัน
```
Dot(A, B) = a₁×b₁ + a₂×b₂ + a₃×b₃ + ...

ค่า:  -∞ ← 0 → +∞ (ยิ่งสูง = ยิ่งคล้าย)
```

**3️⃣ L2 Distance (Euclidean)** — วัด "ระยะห่าง" ระหว่าง 2 จุด
```
L2(A, B) = √( (a₁-b₁)² + (a₂-b₂)² + (a₃-b₃)² + ... )

ค่า:  0 (เหมือนกัน) → +∞ (ต่างกันมาก)  ⚠️ ยิ่งต่ำ = ยิ่งคล้าย
```

---

### 🤔 เมื่อไหร่ใช้อะไร?

**1️⃣ Cosine Similarity — ใช้บ่อยที่สุดใน RAG / NLP**

สนใจแค่ "ทิศทาง" ไม่สนใจ "ขนาด" ของ vector

```
ตัวอย่าง: เปรียบเทียบบทความ
  บทความ A: เขียนเรื่อง AI ยาว 10 หน้า   → vector ยาว (ขนาดใหญ่)
  บทความ B: เขียนเรื่อง AI ยาว 1 ย่อหน้า → vector สั้น (ขนาดเล็ก)

  Cosine = 0.95 ✅ → ทั้งสอง "พูดเรื่องเดียวกัน" ถึงจะยาวต่างกัน
  Dot Product = ค่าต่างกันมาก ❌ → เพราะขนาด vector ต่างกัน
```
👉 **ใช้ตอน**: ค้นหาเอกสารที่เนื้อหาคล้ายกัน, RAG retrieval, semantic search

---

**2️⃣ Dot Product — ใช้เมื่อ vector ถูก normalize แล้ว**

ถ้า normalize vector ให้มีความยาว = 1 แล้ว → Dot Product = Cosine (ผลเท่ากัน แต่เร็วกว่า)

```
ตัวอย่าง: ระบบ recommendation
  User vector (normalize):    [0.5, 0.7, 0.1]  (ขนาด = 1)
  Product A vector (normalize): [0.4, 0.8, 0.2]  (ขนาด = 1)

  Dot = 0.5×0.4 + 0.7×0.8 + 0.1×0.2 = 0.78 ← คำนวณเร็ว!
```
👉 **ใช้ตอน**: ต้องการความเร็วสูง, vector ถูก normalize แล้ว, ระบบ production ขนาดใหญ่

---

**3️⃣ L2 Distance — ใช้เมื่อ "ขนาด" สำคัญ**

วัดระยะห่างจริงๆ ในปริภูมิ เหมือนวัดระยะบนแผนที่

```
ตัวอย่าง: จับกลุ่มภาพที่คล้ายกัน (Image Clustering)
  ภาพแมว A: [0.2, 0.8, 0.3]
  ภาพแมว B: [0.3, 0.7, 0.4]  → L2 = 0.17 (ใกล้กัน ✅)
  ภาพรถยนต์: [0.9, 0.1, 0.8]  → L2 = 1.12 (ไกลกัน ❌)
```
👉 **ใช้ตอน**: image similarity, clustering, anomaly detection

---

> 💡 **สรุปง่ายๆ**: ถ้าทำ RAG → ใช้ **Cosine Similarity** เป็นตัวเลือกแรกเสมอ!

In [ ]:
%%time
# === วัดความคล้ายคลึง 3 วิธี ===
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

labels = ['Q: AI คืออะไร', 'P: AI สาขา CS', 'P: ML ส่วนของ AI', 'P: อากาศดี', 'P: VectorDB']

# 1️⃣ Cosine Similarity
cos_sim = cosine_similarity(embeddings)

# 2️⃣ Dot Product
dot_prod = np.dot(embeddings, embeddings.T)

# 3️⃣ L2 Distance (Euclidean)
l2_dist = euclidean_distances(embeddings)

# --- แสดงผล Cosine Similarity ---
print('1️⃣ Cosine Similarity (ยิ่งสูง = ยิ่งคล้าย, max=1.0)')
print('=' * 82)
print(f'{"":>16}', end='')
for l in labels:
    print(f'{l:>16}', end='')
print()
for i, label in enumerate(labels):
    print(f'{label:>16}', end='')
    for j in range(len(labels)):
        s = cos_sim[i][j]
        marker = ' ⭐' if s > 0.85 and i != j else ''
        print(f'{s:>13.3f}{marker:>3}', end='')
    print()

# --- แสดงผล Dot Product ---
print(f'\n\n2️⃣ Dot Product (ยิ่งสูง = ยิ่งคล้าย, ไม่มี max)')
print('=' * 82)
print(f'{"":>16}', end='')
for l in labels:
    print(f'{l:>16}', end='')
print()
for i, label in enumerate(labels):
    print(f'{label:>16}', end='')
    for j in range(len(labels)):
        print(f'{dot_prod[i][j]:>16.1f}', end='')
    print()

# --- แสดงผล L2 Distance ---
print(f'\n\n3️⃣ L2 Distance (ยิ่งต่ำ = ยิ่งคล้าย, min=0.0)')
print('=' * 82)
print(f'{"":>16}', end='')
for l in labels:
    print(f'{l:>16}', end='')
print()
for i, label in enumerate(labels):
    print(f'{label:>16}', end='')
    for j in range(len(labels)):
        d = l2_dist[i][j]
        marker = ' ⭐' if d < 0.6 and i != j else ''
        print(f'{d:>13.3f}{marker:>3}', end='')
    print()

print('\n💡 สังเกต:')
print('  - Cosine: ข้อความ AI มี similarity สูง (~0.9), อากาศต่ำ (~0.7)')
print('  - L2: ข้อความ AI มีระยะห่างน้อย, อากาศ มีระยะห่างมาก')
print('  - Dot Product ให้แนวโน้มเดียวกับ Cosine เพราะ vector ถูก normalize แล้ว')

### 🧪 แบบฝึกหัด 1.5

ลองสร้างประโยคภาษาไทยของตัวเอง 5 ประโยค แล้วคำนวณ similarity เปรียบเทียบ

In [ ]:
%%time
# 🧪 แบบฝึกหัด: ทดลอง embedding กับประโยคภาษาไทยของตัวเอง
# TODO: เขียนโค้ดของคุณที่นี่



---
## 🔀 Section 1.6: Hybrid Embedding (Dense + Sparse)

### Dense Embedding คืออะไร?

**Dense Embedding** คือการแปลงข้อความเป็น vector ที่ **ทุกช่องมีค่า** (ไม่มีค่าเป็น 0)
โดยใช้ Neural Network เรียนรู้ "ความหมาย" ของข้อความ

```
"แมวนอนบนโซฟา" → [0.12, -0.34, 0.56, 0.78, 0.23, -0.11, ...] (1024 ค่า, ทุกค่าไม่เป็น 0)
```

✅ **เก่ง**: เข้าใจ "ความหมาย" — รู้ว่า "หมา" กับ "สุนัข" คือสิ่งเดียวกัน
❌ **อ่อน**: อาจหา keyword เฉพาะไม่เจอ เช่น รหัสสินค้า "SKU-12345"

---

### Sparse Embedding คืออะไร?

**Sparse Embedding** คือการแปลงข้อความเป็น vector ที่ **ส่วนใหญ่เป็น 0**
มีค่าเฉพาะที่ตำแหน่งของคำที่ปรากฏ โดยใช้วิธีทางสถิติ เช่น BM25 หรือ TF-IDF

```
สมมติ vocabulary = [แมว, หมา, นอน, วิ่ง, โซฟา, สวน, ...] (50,000 คำ)

"แมวนอนบนโซฟา" → [0.8, 0, 0.5, 0, 0.3, 0, 0, 0, 0, ...] (50,000 ค่า, ส่วนใหญ่เป็น 0)
                    แมว  หมา นอน วิ่ง โซฟา สวน
```

✅ **เก่ง**: จับ keyword ตรงๆ ได้แม่น เช่น ชื่อเฉพาะ, รหัส, คำศัพท์เทคนิค
❌ **อ่อน**: ไม่เข้าใจ synonym — คิดว่า "หมา" กับ "สุนัข" เป็นคนละคำ

---

### 🔀 Hybrid Search — ทำไมต้องผสม?

| สถานการณ์ | Dense | Sparse | Hybrid |
|-----------|-------|--------|--------|
| "AI คืออะไร" → หา "ปัญญาประดิษฐ์" | ✅ เจอ | ❌ ไม่เจอ | ✅ เจอ |
| หารหัส "SKU-12345" | ❌ ไม่เจอ | ✅ เจอ | ✅ เจอ |
| "วิธีรักษาโรคเบาหวาน" | ✅ เจอ | ✅ เจอ | ✅✅ เจอดีกว่า |

**สูตร Hybrid:**
```
Hybrid Score = α × Dense Score + (1-α) × Sparse Score

α = 1.0 → ใช้ Dense อย่างเดียว
α = 0.5 → ผสมครึ่งครึ่ง (แนะนำ)
α = 0.0 → ใช้ Sparse อย่างเดียว
```

### 🇹🇭 ทำไมภาษาไทยต้องใช้ Tokenizer พิเศษ?

ภาษาไทย **ไม่มีเว้นวรรคระหว่างคำ** ต่างจากภาษาอังกฤษ:

```
English:  "I love machine learning"
          → แค่ split เว้นวรรค → ["I", "love", "machine", "learning"]  ✅ ง่าย!

Thai:     "ฉันรักการเรียนรู้ของเครื่อง"
          → split เว้นวรรค → ["ฉันรักการเรียนรู้ของเครื่อง"]  ❌ ได้คำเดียว!
          → ใช้ pythainlp  → ["ฉัน", "รัก", "การ", "เรียนรู้", "ของ", "เครื่อง"]  ✅
```

**BM25 ต้องการ token (คำ) เพื่อนับความถี่** → ถ้าไม่ตัดคำ จะหาคำไม่เจอ!

เราใช้ `pythainlp.word_tokenize()` ซึ่งใช้ dictionary + ML ตัดคำภาษาไทย

In [ ]:
%%time
from rank_bm25 import BM25Okapi
import pythainlp

# เตรียมข้อความสำหรับทดสอบ
documents = [
    'ปัญญาประดิษฐ์ คือสาขาที่สร้างระบบอัจฉริยะ',
    'Machine Learning เป็นส่วนหนึ่งของ AI ที่เรียนรู้จากข้อมูล',
    'NLP ช่วยให้คอมพิวเตอร์เข้าใจภาษามนุษย์',
    'Vector Database เก็บข้อมูลในรูปแบบเวกเตอร์',
    'RAG คือเทคนิครวม LLM กับการค้นหาข้อมูล',
    'Qdrant เป็น open-source vector database',
]

# ตัดคำภาษาไทยด้วย PyThaiNLP
tokenized_docs = [pythainlp.word_tokenize(doc) for doc in documents]
print('📝 ตัวอย่างการตัดคำ:')
for i, (doc, tokens) in enumerate(zip(documents[:2], tokenized_docs[:2])):
    print(f'  [{i}] {doc}')
    print(f'      → {tokens}')

# สร้าง BM25 index (Sparse)
bm25 = BM25Okapi(tokenized_docs)
print(f'\n✅ สร้าง BM25 index สำเร็จ ({len(documents)} documents)')

In [ ]:
%%time
# ฟังก์ชัน Hybrid Search
def hybrid_search(query, documents, embed_model, bm25, tokenized_docs, alpha=0.5, top_k=3):
    """ค้นหาแบบ Hybrid: ผสม Dense + Sparse"""
    # Dense search (Semantic)
    q_emb = embed_model.encode([f'query: {query}'])
    doc_embs = embed_model.encode([f'passage: {d}' for d in documents])
    dense_scores = cosine_similarity(q_emb, doc_embs)[0]
    
    # Sparse search (BM25)
    q_tokens = pythainlp.word_tokenize(query)
    sparse_scores = bm25.get_scores(q_tokens)
    
    # Normalize scores to [0, 1]
    if max(dense_scores) > 0:
        dense_norm = dense_scores / max(dense_scores)
    else:
        dense_norm = dense_scores
    
    if max(sparse_scores) > 0:
        sparse_norm = sparse_scores / max(sparse_scores)
    else:
        sparse_norm = sparse_scores
    
    # Hybrid score
    hybrid_scores = alpha * dense_norm + (1 - alpha) * sparse_norm
    
    # เรียงลำดับ
    ranked = sorted(enumerate(zip(documents, dense_norm, sparse_norm, hybrid_scores)),
                    key=lambda x: x[1][3], reverse=True)
    return ranked[:top_k]

# ทดสอบ
query = 'AI เรียนรู้จากข้อมูล'
results = hybrid_search(query, documents, embed_model, bm25, tokenized_docs, alpha=0.5)

print(f'🔍 Query: "{query}"')
print(f'{"":>4}{"Document":<50} {"Dense":>8} {"Sparse":>8} {"Hybrid":>8}')
print('=' * 82)
for idx, (doc, dense, sparse, hybrid) in results:
    print(f'  [{idx}] {doc:<48} {dense:>8.3f} {sparse:>8.3f} {hybrid:>8.3f}')

In [ ]:
%%time
# เปรียบเทียบ alpha ต่างๆ
print('📊 เปรียบเทียบ alpha (Dense weight) ต่างๆ:')
print('=' * 70)
query = 'vector database สำหรับ RAG'
print(f'Query: "{query}"\n')

for alpha in [0.0, 0.3, 0.5, 0.7, 1.0]:
    results = hybrid_search(query, documents, embed_model, bm25, tokenized_docs, alpha=alpha, top_k=2)
    top_doc = results[0][1][0]
    top_score = results[0][1][3]
    label = 'Sparse only' if alpha == 0 else 'Dense only' if alpha == 1 else f'Hybrid'
    print(f'  α={alpha:.1f} ({label:<12}): [{results[0][0]}] {top_doc[:45]}... ({top_score:.3f})')

### 🧪 แบบฝึกหัด 1.6

ลองเปลี่ยน query เป็นภาษาไทยล้วนๆ และภาษาอังกฤษล้วนๆ แล้วดูว่า alpha ค่าไหนให้ผลดีที่สุด

In [ ]:
%%time
# 🧪 แบบฝึกหัด: ทดลอง Hybrid search กับ query ภาษาไทย
# TODO: เขียนโค้ดของคุณที่นี่



---
## 🗄️ Section 1.7: ติดตั้ง Qdrant VectorDB

### Qdrant คืออะไร?

**[Qdrant](https://qdrant.tech/)** เป็น open-source vector database ที่:
- รองรับทั้ง **Dense** และ **Sparse** vectors
- มี **Hybrid Search** ในตัว
- ใช้ **Rust** เขียน → เร็วมาก
- รองรับ **filtering** ด้วย metadata

ในวันนี้เราจะใช้ **in-memory mode** บน Colab (ไม่ต้องติดตั้ง server)

In [ ]:
%%time
from qdrant_client import QdrantClient, models

# สร้าง Qdrant client แบบ in-memory
qdrant = QdrantClient(':memory:')  # ใช้ RAM แทน disk

# กำหนดค่า
COLLECTION_NAME = 'thai_rag_workshop'
VECTOR_SIZE = embed_model.get_sentence_embedding_dimension()  # 1024 สำหรับ e5-large

print(f'✅ สร้าง Qdrant client สำเร็จ (in-memory mode)')
print(f'📏 Vector size: {VECTOR_SIZE} มิติ')

In [ ]:
%%time
# สร้าง Collection
qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(
        size=VECTOR_SIZE,
        distance=models.Distance.COSINE,  # ใช้ Cosine similarity
    ),
)

# ตรวจสอบ
info = qdrant.get_collection(COLLECTION_NAME)
print(f'✅ สร้าง Collection "{COLLECTION_NAME}" สำเร็จ!')
print(f'  Status: {info.status}')
print(f'  Vector size: {info.config.params.vectors.size}')
print(f'  Distance: {info.config.params.vectors.distance}')
print(f'  Points: {info.points_count}')

### 🧪 แบบฝึกหัด 1.7

ลองสร้าง collection ใหม่ชื่อ `my_collection` ด้วย distance metric แบบ `EUCLID`

In [ ]:
# 🧪 แบบฝึกหัด: สร้าง collection ของตัวเอง
# TODO: เขียนโค้ดของคุณที่นี่



---
## 📥 Section 1.8: สร้าง Index (Upsert ข้อมูล)

### ขั้นตอนการ Index

```
1. อ่านเอกสาร → 2. Chunk → 3. Embed → 4. Upsert เข้า Qdrant
```

เราจะนำข้อมูลที่เตรียมไว้ (chunk + embed) มาใส่ใน Qdrant

In [ ]:
%%time
# เตรียมข้อมูลสำหรับ indexing
# อ่านเอกสารทั้งหมด → chunk ด้วย recursive splitter

all_chunks = []
for fname in sorted(os.listdir('data')):
    if not fname.endswith('.txt'):
        continue
    with open(f'data/{fname}', 'r', encoding='utf-8') as f:
        text = f.read()
    
    chunks = recursive_splitter.split_text(text)
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            'text': chunk,
            'source': fname,
            'chunk_index': i,
        })

print(f'📊 เตรียมข้อมูลสำเร็จ:')
print(f'  จำนวน chunks ทั้งหมด: {len(all_chunks)}')
for fname in sorted(os.listdir('data')):
    if fname.endswith('.txt'):
        count = sum(1 for c in all_chunks if c['source'] == fname)
        print(f'  {fname}: {count} chunks')

In [ ]:
%%time
# สร้าง embeddings สำหรับทุก chunk
print('⏳ กำลังสร้าง embeddings...')
chunk_texts = [f"passage: {c['text']}" for c in all_chunks]
chunk_embeddings = embed_model.encode(chunk_texts, show_progress_bar=True)
print(f'✅ สร้าง embeddings สำเร็จ! ({len(chunk_embeddings)} vectors × {chunk_embeddings.shape[1]} มิติ)')

In [ ]:
%%time
# Upsert เข้า Qdrant
points = []
for i, (chunk, embedding) in enumerate(zip(all_chunks, chunk_embeddings)):
    points.append(
        models.PointStruct(
            id=i,
            vector=embedding.tolist(),
            payload={
                'text': chunk['text'],
                'source': chunk['source'],
                'chunk_index': chunk['chunk_index'],
            }
        )
    )

# Upsert ทีเดียว
qdrant.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
)

# ตรวจสอบ
info = qdrant.get_collection(COLLECTION_NAME)
print(f'✅ Upsert สำเร็จ! จำนวน points ใน collection: {info.points_count}')

### 🧪 แบบฝึกหัด 1.8

ลองเพิ่มข้อมูลใหม่เข้าไปใน collection (เช่น ข้อความเกี่ยวกับหัวข้อที่สนใจ)

In [ ]:
# 🧪 แบบฝึกหัด: เพิ่มข้อมูลของตัวเอง
# TODO: เขียนโค้ดของคุณที่นี่



---
## 🔎 Section 1.9: Retrieve ข้อมูลจาก Qdrant

มาทดสอบค้นหาข้อมูลจาก Vector Database ที่เราสร้างไว้!

### Dense Search (Semantic Search)

### 🎯 top_k คืออะไร?

**`top_k`** คือจำนวนผลลัพธ์ที่ต้องการให้ระบบส่งกลับมา โดยเรียงตาม score จากมากไปน้อย

```
ตัวอย่าง: ถ้ามี 100 chunks ใน database

top_k=3  → ได้ 3 chunks ที่คล้ายที่สุด ✅ เร็ว, บริบทกระชับ
top_k=5  → ได้ 5 chunks ที่คล้ายที่สุด    บริบทมากขึ้น
top_k=10 → ได้ 10 chunks ที่คล้ายที่สุด   ⚠️ อาจมี noise ปนมา
```

**การเลือก top_k ที่เหมาะสม:**

| top_k | เหมาะกับ | ข้อดี | ข้อเสีย |
|-------|---------|-------|--------|
| 1-3 | คำถามเฉพาะเจาะจง | เร็ว, ตรงประเด็น | อาจพลาดข้อมูลสำคัญ |
| 3-5 | คำถามทั่วไป (แนะนำ) | สมดุลดี | — |
| 5-10 | คำถามกว้าง / ซับซ้อน | ครอบคลุม | ช้าขึ้น, context ยาว |

> 💡 **ในทางปฏิบัติ**: top_k=3~5 เป็นค่าที่นิยมใน RAG
> เพราะ LLM ทำงานได้ดีกับบริบทที่กระชับ ไม่ยาวเกินไป

In [ ]:
%%time
def search_qdrant(query, top_k=3):
    """ค้นหาข้อมูลจาก Qdrant ด้วย Dense embedding"""
    # สร้าง query embedding
    q_embedding = embed_model.encode([f'query: {query}'])[0]
    
    # ค้นหา
    results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=q_embedding.tolist(),
        limit=top_k,
        with_payload=True,
    )
    return results

# ทดสอบค้นหา
query = 'ปัญญาประดิษฐ์คืออะไร'
results = search_qdrant(query)

print(f'🔍 Query: "{query}"')
print(f'📊 ผลลัพธ์ Top-3:')
print('=' * 70)
for i, point in enumerate(results.points):
    print(f'\n🏆 อันดับ {i+1} (Score: {point.score:.4f})')
    print(f'   แหล่งที่มา: {point.payload["source"]} (chunk {point.payload["chunk_index"]})')
    print(f'   เนื้อหา: {point.payload["text"][:150]}...')

### ลองค้นหาคำถามอื่นๆ

In [ ]:
%%time
# ทดสอบหลายคำถาม
test_queries = [
    'Vector Database คืออะไร',
    'RAG ทำงานอย่างไร',
    'Deep Learning กับ Machine Learning ต่างกันอย่างไร',
]

for query in test_queries:
    results = search_qdrant(query, top_k=2)
    print(f'\n🔍 Query: "{query}"')
    for i, point in enumerate(results.points):
        print(f'   [{i+1}] (Score: {point.score:.4f}) {point.payload["text"][:80]}...')
    print('-' * 70)

### ค้นหาพร้อม Filter ด้วย Metadata

In [ ]:
%%time
# ค้นหาเฉพาะจากเอกสารที่กำหนด
query = 'การเรียนรู้ของเครื่อง'
q_embedding = embed_model.encode([f'query: {query}'])[0]

results_filtered = qdrant.query_points(
    collection_name=COLLECTION_NAME,
    query=q_embedding.tolist(),
    query_filter=models.Filter(
        must=[
            models.FieldCondition(
                key='source',
                match=models.MatchValue(value='case_healthcare.txt'),
            )
        ]
    ),
    limit=3,
    with_payload=True,
)

print(f'🔍 Query: "{query}" (filter: source=case_healthcare.txt)')
for i, point in enumerate(results_filtered.points):
    print(f'   [{i+1}] ({point.score:.4f}) [{point.payload["source"]}] {point.payload["text"][:80]}...')

### 🧪 แบบฝึกหัด 1.9

1. ลองค้นหาคำถามภาษาไทยอื่นๆ
2. ลองใช้ filter เพื่อค้นหาเฉพาะจากเอกสารที่ต้องการ
3. ลองเปลี่ยนค่า `top_k` แล้วดูผลลัพธ์ที่ต่างกัน

In [ ]:
# 🧪 แบบฝึกหัด: ทดลองค้นหาจาก Qdrant
# TODO: เขียนโค้ดของคุณที่นี่



---
## 🎯 สรุป: สิ่งที่เราเรียนรู้ในวันนี้

### Pipeline ทั้งหมดของ Data Engineering สำหรับ RAG

```
📄 Raw Documents
    │
    ├── 🔍 1.1 ตรวจ Duplicate (MD5) → ลบไฟล์ซ้ำ
    │
    ├── 📝 1.3/1.4 แปลง Markdown (Gemini / Docling)
    │
    ├── ✂️ 1.2 Chunking (Fixed / Recursive / Sentence)
    │
    ├── 🔢 1.5 Dense Embedding (multilingual-e5-large)
    │
    ├── 🔀 1.6 Hybrid Embedding (Dense + BM25)
    │
    ├── 🗄️ 1.7 สร้าง Qdrant Collection
    │
    ├── 📥 1.8 Index (Upsert vectors + metadata)
    │
    └── 🔎 1.9 Retrieve (Dense / Filtered search)
```

### 🔑 Key Takeaways

1. **คุณภาพข้อมูลสำคัญที่สุด** — "Garbage in, garbage out"
2. **Chunking strategy** มีผลมากต่อคุณภาพ RAG
3. **Hybrid search** (Dense + Sparse) มักดีกว่าใช้อย่างใดอย่างหนึ่ง
4. **Metadata** ช่วยให้ filter ผลลัพธ์ได้แม่นยำขึ้น

### 📅 ครั้งหน้า: Day 2 — Building Agents: Finding the Needle in the Haystack

เราจะเรียนรู้วิธีสร้าง **AI Agents** ที่ใช้ RAG pipeline ที่สร้างไว้
ในการตอบคำถามอย่างชาญฉลาด!